In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, confusion_matrix

df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\5thsem_ojt\project1_diabetes\data\processed\hebin_p1w2_features.csv')


feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi_category","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [2]:
candidates = {
    "LogReg": (
        "scaled",
        LogisticRegression(max_iter=1000)
    ),
    "Decision Tree": (
        "raw",
        DecisionTreeClassifier(max_depth=3, random_state=42)
    ),
    "KNN": (
        "scaled",
        KNeighborsClassifier(n_neighbors=25)
    ),
    "SVC balanced": (
        "scaled",
        SVC(class_weight="balanced", random_state=42)
    ),
    "RF balanced": (
        "raw",
        RandomForestClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=42
        )
    )
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50  # Week 5 decision

In [3]:
print ('rows:',len(df),"| train:",len(x_train),"| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("featres after get dummies:",x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)

rows: 950 | train: 760 | test: 190
diabetic in test set: 55
featres after get dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


In [4]:
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        x, y,
        test_size=0.2,
        random_state=rs,
        stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)  

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))

        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)
        row[name]=round(recall_score(yb,pred),3)
    rows.append(row)  

splits=pd.DataFrame(rows).set_index('split')
print('splits')  
spread=(splits.max() - splits.min()).round(3)
print('\nspread (best - worst):\n', spread.sort_values(ascending=False))

splits

spread (best - worst):
 Decision Tree    0.291
SVC balanced     0.145
RF balanced      0.091
LogReg           0.073
KNN              0.073
dtype: float64


# CROSS VALIDATION


K-Fold Cross-Validation is a technique used to check how well a machine learning model performs on different parts of a dataset.

In [7]:
from sklearn.model_selection import StratifiedKFold

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i,(tr,te) in enumerate(cv.split(x_train,y_train),1):
    print(f'fold{i}: train {len(tr)} score{len(te)}'
          f'diabetic in score fold{y_train.iloc[te].sum()}')

def make_model(kind, model):
    """Scaled models get the scaler INSIDE a Pipeline,so it refits per fold."""
    if kind == 'scaled':
        return Pipeline([("scale",StandardScaler()),('model',model)])
    return model

print()
for name,(kind,model)in candidates.items():
    print(f"{name:15s}->{type(make_model(kind,model)).__name__}")

fold1: train 608 score152diabetic in score fold43
fold2: train 608 score152diabetic in score fold43
fold3: train 608 score152diabetic in score fold44
fold4: train 608 score152diabetic in score fold44
fold5: train 608 score152diabetic in score fold44

LogReg         ->Pipeline
Decision Tree  ->DecisionTreeClassifier
KNN            ->Pipeline
SVC balanced   ->Pipeline
RF balanced    ->RandomForestClassifier


Setup cros validation the right way

Cross validate every candidate

In [8]:
from sklearn.model_selection import cross_validate

rows=[]
for name, (kind,model) in candidates.items():
    est=make_model(kind,model)
    res=cross_validate(est, x_train,y_train,cv=cv,
                       scoring=['recall','f1'])
    single=clone(est).fit(x_train,y_train)
    rows.append({
        'model':    name,
        'cv_recall': round(res["test_recall"].mean(),3),
        'cv_std':  round(res["test_recall"].std(),3),
        'cv_f1':   round(res["test_f1"].mean(),3),
        'week6_recall': round(recall_score(y_test,single.predict(x_test)),3),
        'folds':  np.round(res['test_recall'],3).tolist(),
    })

cv_table=pd.DataFrame(rows).sort_values('cv_recall',ascending=False)
print(cv_table.to_string(index=False))

        model  cv_recall  cv_std  cv_f1  week6_recall                               folds
 SVC balanced      0.721   0.046  0.559         0.618 [0.744, 0.791, 0.682, 0.727, 0.659]
  RF balanced      0.652   0.051  0.521         0.655 [0.721, 0.651, 0.682, 0.636, 0.568]
       LogReg      0.303   0.073  0.390         0.291  [0.419, 0.326, 0.318, 0.25, 0.205]
Decision Tree      0.216   0.076  0.296         0.291 [0.116, 0.326, 0.159, 0.205, 0.273]
          KNN      0.174   0.018  0.261         0.182 [0.186, 0.163, 0.159, 0.205, 0.159]
